<a href="https://colab.research.google.com/github/prudhvi260/Scrapy/blob/main/Scrapy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Scrapy
!pip install scrapy
import scrapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.8/259.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 5.5 MB/s eta 0:00:00


In [ ]:
class QuotesSpider(scrapy.Spider):
    name = "worldmeters"
    allowed_domains = ["www.worldometers.info"]
    start_urls = [
        'https://www.worldometers.info/world-population/population-by-country/'
    ]

    def parse(self, response):
      #title = response.xpath('//h1/text()').get()
      countries = response.xpath('//td/a')

      for country in countries:
        name = country.xpath('.//text()').get()
        link = country.xpath('.//@href').get()
        yield {'name': name, 'link': link}
        #absolute_url = f'https://www.worldometers.info/{link}'
        #absolute_url = response.urljoin(link)
        #yield scrapy.Request(url=absolute_url)
        yield response.follow(url=link, callable=self.parse_country, meta={'country':name})

      #yield {'title': title, 'countries': countries}

    def parse_country(self, response):
      country = response.request.meta['country']
      #response.xpath('(//table[@class="table table-striped table-bordered table-hover table-condensed table-list"])[1]/tbody/tr')
      rows = response.xpath("(//table[contains(@class,'table')])[1]/tbody/tr")
      for row in rows:
        year = row.xpath('.//td[1]/text()').get()
        population = row.xpath('.//td[2]/strong/text()').get()
        yield {'country':country,'year': year, 'population': population}





In [ ]:
#Pagination with scrapy

class AudiSpider(scrapy.Spider):
    name = "audible"
    allowed_domains = ["www.audible.com"]
    # start_urls = [
    #     'https://www.audible.com/search/'
    # ]

    def start_requests(self):
      yield scrapy.Request(url='https://www.audible.com/search/',callback=self.parse, headers={''})

    def parse(self, response):
      product_container = response.xpath('//div[@class="adbl-impression-container "]//li[contains(@class, "productListItem")]')

      for product in product_container:
        book_title = product.xpath('.//h3[contains(@class,"bc-heading")]/a/text()').get()
        book_author = product.xpath(".//li[contains(@class,'authorLabel')]/span/a/text()").getall()
        book_length = product.xpath(".//li[contains(@class,'runtimeLabel')]/span/text()").get()

        yield {'title': book_title, 'author': book_author, 'length': book_length, 'user-Agent':response.request.headers['User-Agent'],}

        pagination = response.xpath('//ul[contains(@class,"pagingElements")]')

        next_page_url = pagination.xpath('.//span[contains(@class,"nextButton")]/a/@href').get()

        if next_page_url:
          yield response.follow(url=next_page_url,callback=self.parse, headers={''})
